# Display Rendering — Tonemapping an HDR image for sRGB display

The camera-characterized image is **HDR** (values can exceed 1.0, covering 10+ stops of dynamic range).
A standard sRGB display can only show values in 0…1. We need **tonemapping** to compress
the HDR range into the display range while preserving as much perceptual detail as possible.

We build the tonemapping pipeline step by step:
1. **Exposure** — shift brightness in log space
2. **Contrast** — expand or compress the tonal range around the image median
3. **Shadow lift** — raise the black point slightly
4. **Highlight compression** — smoothly compress overexposed highlights using $x/(1+x)$
5. **Luminance-only tonemapping** — apply the curve to intensity, not R/G/B independently

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

linear2sRGB = lambda x: np.where(x > 0.0031308, 1.055 * np.clip(x, 0, None) ** (1/2.4) - 0.055, 12.92 * x)
clamp       = lambda x: np.clip(x, 0, 1)

data = np.load('23_RGB_CamCharacterized_for_sRGB.npz')
hdr  = data['hdr_psInv_img']

# Work with a half-resolution copy for faster processing
from PIL import Image
rgb = np.array(Image.fromarray(
    np.clip(hdr, 0, None).astype(np.float32)  # PIL needs float32
).resize((hdr.shape[1]//2, hdr.shape[0]//2), Image.BILINEAR))
rgb = np.maximum(0, rgb)   # ensure no negative values

print(f'Working image shape: {rgb.shape}')
print(f'Value range: {rgb.min():.3f} … {rgb.max():.3f}')

## Helper: plot a tone curve

We visualise each tonemapping step with a log-scale input plot.

In [ ]:
def plot_tone_curve(tone_fn, title='Tone curve'):
    """Plot a tonemapping function on a log-scale x-axis (−12 to +2 stops)."""
    x = 2.0 ** np.linspace(-12, 2, 500)
    y = clamp(tone_fn(x))
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.semilogx(x, y, lw=2)
    ax.set_xlabel('Input value (log scale, stops)')
    ax.set_ylabel('Output (0…1)')
    ax.set_title(title)
    ax.grid(True, which='both', alpha=0.4)
    ax.set_xlim(2**-12, 2**2)
    ax.set_ylim(0, 1)
    # Mark stop lines
    for s in range(-12, 3):
        ax.axvline(2**s, color='gray', lw=0.5, alpha=0.5)
    plt.tight_layout()
    plt.show()

## Step 1 — Exposure

Exposure is a multiplicative shift in linear light, equivalent to adding stops in log space:

$$I_{\text{out}} = I_{\text{in}} \cdot 2^{\text{exposure}}$$

In [ ]:
def tm_exposure(x, exposure):
    return x * 2**exposure

exposure = 0.5   # stops

plot_tone_curve(lambda x: tm_exposure(x, exposure), f'Exposure: {exposure:+.1f} stops')

fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(linear2sRGB(clamp(tm_exposure(rgb, exposure))))
ax.set_title(f'Exposure {exposure:+.1f} stops')
ax.axis('off')
plt.tight_layout()
plt.show()

## Step 2 — Contrast

We scale the log-space distance from the image median:

$$\log_2 I_{\text{out}} = (\log_2 I_{\text{in}} - \log_2 \tilde{I}) \cdot c + \log_2 \tilde{I}$$

where $\tilde{I}$ is the image median and $c$ is the contrast factor.
$c > 1$ increases contrast, $c < 1$ decreases it.

In [ ]:
img_median = np.median(rgb)

def tm_contrast(x, contrast):
    return 2 ** ((np.log2(np.maximum(x, 1e-10)) - np.log2(img_median)) * contrast
                 + np.log2(img_median))

contrast = 0.8   # try values between 0.5 and 2.0

plot_tone_curve(lambda x: tm_contrast(x, contrast), f'Contrast: {contrast}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(linear2sRGB(clamp(tm_contrast(rgb, contrast))))
ax.set_title(f'Contrast = {contrast}')
ax.axis('off')
plt.tight_layout()
plt.show()

## Step 3 — Combine exposure + contrast

In [ ]:
def tm_combined(x, exposure, contrast):
    return tm_contrast(tm_exposure(x, exposure), contrast)

exposure = 0.5
contrast = 1.3

plot_tone_curve(lambda x: tm_combined(x, exposure, contrast),
                f'Exposure {exposure:+.1f}, Contrast {contrast}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(linear2sRGB(clamp(tm_combined(rgb, exposure, contrast))))
ax.set_title(f'Exposure {exposure:+.1f} + Contrast {contrast}')
ax.axis('off')
plt.tight_layout()
plt.show()

## Step 4 — Shadow lift

Raising the black point slightly reduces the perceived "muddiness" of deep shadows:

$$I_{\text{out}} = I_{\text{in}} + 2^{-s}$$

where $s$ is `stops_below_white` — typically 8…12.

In [ ]:
def tm_shadow_lift(x, stops_below_white):
    return x + 2**(-stops_below_white)

def tm_full(x, exposure, contrast, stops_below_white):
    return tm_shadow_lift(tm_combined(x, exposure, contrast), stops_below_white)

exposure         = 0.5
contrast         = 1.3
stops_below_white = 10

plot_tone_curve(lambda x: tm_full(x, exposure, contrast, stops_below_white),
                f'+ Shadow lift at −{stops_below_white} stops')

fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(linear2sRGB(clamp(tm_full(rgb, exposure, contrast, stops_below_white))))
ax.set_title('With shadow lift')
ax.axis('off')
plt.tight_layout()
plt.show()

## Step 5 — Highlight compression

The function $f(x) = x / (1 + x)$ maps $[0, \infty) \to [0, 1)$ smoothly —
highlights are compressed but never clipped:

In [ ]:
def tm_highlight(x):
    return x / (1 + x)

def tm_final(x, exposure, contrast, stops_below_white):
    return tm_highlight(tm_full(x, exposure, contrast, stops_below_white))

exposure         = 0.5
contrast         = 1.3
stops_below_white = 10

plot_tone_curve(lambda x: tm_final(x, exposure, contrast, stops_below_white),
                'Full pipeline with highlight compression')

fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(linear2sRGB(clamp(tm_final(rgb, exposure, contrast, stops_below_white))))
ax.set_title('With highlight compression')
ax.axis('off')
plt.tight_layout()
plt.show()

## Step 6 — Apply to luminance only

Applying the tonemapping curve independently to R, G and B shifts colour saturation.
A better approach: compute the **luminance**, tonemap it, then scale all three channels
to match the new luminance while preserving the original colour ratios.

$$I_{\text{out}} = I_{\text{in}} \cdot \frac{L_{\text{ldr}}}{L_{\text{hdr}}}$$

where $L = 0.2126 R + 0.7152 G + 0.0722 B$ (Rec. 709 luminance coefficients).

In [ ]:
exposure         = 0.5
contrast         = 1.3
stops_below_white = 10

# Rec. 709 luminance
hdr_lum = 0.2126 * rgb[:,:,0] + 0.7152 * rgb[:,:,1] + 0.0722 * rgb[:,:,2]
ldr_lum = clamp(tm_final(hdr_lum, exposure, contrast, stops_below_white))

# Scale RGB by the lum ratio; avoid division by zero
lum_ratio       = ldr_lum / np.maximum(hdr_lum, 1e-10)
tonemapped_lum  = rgb * lum_ratio[:, :, np.newaxis]

# Per-channel (naive) tonemapping for comparison
tonemapped_rgb  = clamp(tm_final(rgb, exposure, contrast, stops_below_white))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(linear2sRGB(tonemapped_rgb))
axes[0].set_title('Per-channel tonemapping (colours shift)')
axes[0].axis('off')

axes[1].imshow(linear2sRGB(clamp(tonemapped_lum)))
axes[1].set_title('Luminance-only tonemapping (colours preserved)')
axes[1].axis('off')

plt.suptitle('Compare: which approach preserves colour saturation better?')
plt.tight_layout()
plt.show()